In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Data Preparation
data_dir = "./COVID-19_Radiography_Dataset"
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset_full = datasets.ImageFolder(root=data_dir, transform=train_transforms)
test_dataset_full = datasets.ImageFolder(root=data_dir, transform=test_transforms)

class_mapping = train_dataset_full.class_to_idx
print(f"Class Encoding: {class_mapping}")

targets = train_dataset_full.targets
indices = list(range(len(train_dataset_full)))

train_indices, temp_indices, train_targets, temp_targets = train_test_split(
    indices, targets, test_size=0.40, stratify=targets, random_state=42
)

val_indices, test_indices, _, _ = train_test_split(
    temp_indices, temp_targets, test_size=0.50, stratify=temp_targets, random_state=42
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(test_dataset_full, val_indices)
test_dataset = Subset(test_dataset_full, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data Splitted -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("DataLoaders are ready!")


#FFNN Model (with Conv Feature Extractor)
class FFNN_COVID(nn.Module):
    def __init__(self, num_classes=2):
        super(FFNN_COVID, self).__init__()
        
        # Convolutional Feature Extractor (in_channels=1)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU(inplace=True)
        self.dropout2d = nn.Dropout2d(0.25)
        
        # FFNN Part
        self.flatten = nn.Flatten()
        
       # 16384
        self.fc1 = nn.Linear(8192, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, num_classes)
        
        self.bn1 = nn.BatchNorm1d(4096)
        self.bn2 = nn.BatchNorm1d(1024)
        self.bn3 = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # x: [Batch, 1, 128, 128]
        
        x = self.relu(self.conv1(x))
        x = self.pool(x)                    # 64x64
        x = self.dropout2d(x)
        
        x = self.relu(self.conv2(x))
        x = self.pool(x)                    # 32x32
        x = self.dropout2d(x)
        
        x = self.relu(self.conv3(x))
    
        
        x = F.adaptive_avg_pool2d(x, (8, 8))   
        
        # (Flatten = 16384)
        x = self.flatten(x)
        
        # FFNN
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        
        x = self.fc4(x)
        
        return x


# Model
num_classes = len(class_mapping)
model = FFNN_COVID(num_classes=num_classes)

print(f"\nModel Created Successfully!")
print(f"Number of Classes: {num_classes}")
print(f"Expected Flatten Size: 16384")

# test output
dummy_input = torch.randn(2, 1, 128, 128)
output = model(dummy_input)
print(f"Output Shape: {output.shape}")

Class Encoding: {'COVID': 0, 'Lung_Opacity': 1, 'Normal': 2, 'Viral Pneumonia': 3}
Data Splitted -> Train: 25398 | Val: 8466 | Test: 8466
DataLoaders are ready!

Model Created Successfully!
Number of Classes: 4
Expected Flatten Size: 16384
Output Shape: torch.Size([2, 4])
